# Appendix — High-fidelity (dynamical) diffraction with abTEM

**What this is.** The main digital twin computes diffraction *kinematically* (a single
scattering event, `|Σ_j f_j e^{2πi q·r_j}|²`). That is fast and gives correct spot
*positions*, which is all that acquisition-logic testing needs. It does **not** capture
*dynamical* (multiple-scattering) effects: correct spot *intensities*, thickness
fringes, channelling contrast, Kikuchi lines, or the thermal-diffuse background. Analysis
code that reads those channels (thickness from CBED, atom-counting from intensities,
strain from precise spot shifts, ptychography from the complex exit wave) needs a
dynamical simulation to behave the way it would on **real** data.

This appendix shows how to generate **analysis-grade** diffraction from the *same* atomic
model the twin uses, with the multislice code **abTEM**, and hand the result to an external
tool (**py4DSTEM**) exactly as you would with experimental data. It is deliberately
**CPU-only and self-contained** — no GPU, no dependency on the twin server. It is slow
compared with the kinematical engine, but every pattern here is real multislice physics.

**How it connects to the twin.** The twin's samples expose
`get_atoms_in_region(...) -> (positions_Å, atomic_numbers)`. That is precisely the input a
multislice engine wants (here we build the equivalent FCC atoms directly with ASE so the
appendix stands alone). The intended architecture is a **two-path twin**: the fast
kinematical path stays the interactive default for developing acquisition logic; this
dynamical path is an **offline dataset generator** that produces realistic 4D-STEM data
for someone's analysis/reconstruction code to be validated against — the expensive work is
paid **once at generation**, not per interaction.

**Scope of this appendix (kept small for CPU):**
1. A single dynamical SAED pattern from an FCC (Au) crystal — compare to kinematical.
2. Adding **frozen phonons** (thermal diffuse scattering) — the realistic background.
3. A small **4D-STEM scan** (convergent probe) saved as a portable 4D array.
4. A round-trip check: analyse the simulated 4D data (mean pattern + virtual bright-field)
   the way an external tool would — with plain NumPy (no conflicts), plus an **optional**
   isolated cell showing the identical operations through **py4DSTEM** itself.

> **Packaging note.** The core here (abTEM + ASE) works with Colab's NumPy 2.x — no
> downgrade, no restart. Only the *optional* py4DSTEM cell needs `numpy<2` (py4DSTEM pins
> it), which is why it is isolated at the end. If you saw `numpy 1.26.4 is incompatible`
> errors, they come from installing py4DSTEM alongside Colab's NumPy-2 packages; the core
> notebook avoids that entirely.

> Runtime note: on a typical CPU each step below is seconds to a couple of minutes at the
> sizes chosen here. Realistic sizes (larger cells, 128×128 scans, 16–32 phonon configs)
> are where a GPU becomes worthwhile — that is the natural next step, not required here.


## 0. Install (CPU build) — no version conflicts

The core of this appendix needs only **abTEM** and **ASE**, which work with the NumPy 2.x
that Colab ships — so there is **nothing to downgrade and no runtime restart**.

> **Important:** do *not* `pip install py4dstem` here. py4DSTEM currently pins `numpy<2`,
> which forces a NumPy downgrade and breaks the NumPy-2 packages Colab pre-installs
> (that is the source of the `numpy 1.26.4 is incompatible` errors). We keep py4DSTEM out
> of the core notebook and, in the optional Section 5, do the same analysis with plain
> NumPy — no extra dependency. A separate cell shows the real py4DSTEM call for anyone who
> specifically wants it, with the correct isolated-install instructions.

In [ ]:
# Core dependencies only. These are compatible with Colab's NumPy 2.x.
!pip -q install abtem ase
import abtem, ase, numpy as np, matplotlib.pyplot as plt
print("abtem", abtem.__version__, "| ase", ase.__version__, "| numpy", np.__version__)


def show_diffraction(ax, pattern, title=None, beamstop_radius=6, cmap="inferno"):
    """Display a diffraction pattern with the direct (000) beam suppressed.

    In any diffraction pattern the undiffracted 'direct beam' at the centre is
    ~100x brighter than the Bragg spots, so a plain (even log) display shows only
    that central dot and the spots vanish. We clip the display maximum to the
    Bragg-spot level -- the direct beam saturates (as it does behind a real beam
    stop) and the weaker spots become visible. This changes only the DISPLAY, not
    the data."""
    p = np.asarray(pattern, dtype=float)
    cy, cx = np.unravel_index(np.argmax(p), p.shape)
    Y, X = np.mgrid[0:p.shape[0], 0:p.shape[1]]
    outside = np.hypot(Y - cy, X - cx) > beamstop_radius
    vmax = np.percentile(p[outside], 99.9) if outside.any() else p.max()
    vmax = max(vmax, 1e-12)
    ax.imshow(p, cmap=cmap, vmax=vmax)   # clip the bright direct beam
    ax.axis("off")
    if title:
        ax.set_title(title)


## 1. Build the FCC crystal (same material as the twin: Au, a = 4.05 Å)

We build the atoms directly with ASE so the appendix is standalone. In the integrated
tool you would instead take `positions, Z = sample.get_atoms_in_region(...)` from the twin
and wrap them in an `ase.Atoms` object — the rest of this notebook is identical.

In [ ]:
from ase.build import bulk

# Cubic FCC Au conventional cell, tiled to a thin slab.
#   lateral: 6x6 cells (~24 x 24 A)   thickness: 25 cells (~101 A ~ 10 nm)
a_Au = 4.05
atoms = bulk("Au", "fcc", a=a_Au, cubic=True) * (6, 6, 25)
print(f"{len(atoms)} atoms; cell (A) = {np.round(atoms.cell.lengths(), 1)}")

# --- how you would instead ingest atoms FROM the twin sample -------------------
# from ase import Atoms
# positions_A, Z = sample.get_atoms_in_region(cx_um=0, cy_um=0, half_width_um=0.02, depth_nm=10)
# atoms = Atoms(numbers=Z, positions=positions_A, cell=[Lx, Ly, Lz], pbc=True)
# -------------------------------------------------------------------------------

abtem.show_atoms(atoms, plane="xy", title="FCC Au — projected along the beam [001]")
plt.show()


## 2. A single dynamical SAED pattern (multislice)

We build a projected **potential** (sliced along the beam), send a **plane wave** through
it (multislice), and take the exit-wave **diffraction pattern**. Unlike the kinematical
engine, the spot *intensities* here reflect multiple scattering and the direct-beam
depletion, and they change correctly with thickness.

**About the bright centre.** Every diffraction pattern has a very bright spot in the middle
— the **direct (undiffracted) beam**. It is typically ~100× brighter than the Bragg spots,
so a naive display shows only that central dot and the spots look like they are missing.
They are not: we simply clip the display maximum to the Bragg level (`show_diffraction`
above), which saturates the direct beam — exactly what a physical **beam stop** does — so the
weaker diffracted spots become visible. This is a display choice; the data is unchanged.

In [ ]:
# Potential: 0.08 A sampling, 2 A slices. 'infinite' projection is the standard fast choice.
potential = abtem.Potential(atoms, sampling=0.08, slice_thickness=2.0,
                            projection="infinite", parametrization="lobato")

# 200 kV plane-wave illumination -> exit wave -> diffraction pattern (out to 60 mrad).
plane_wave = abtem.PlaneWave(energy=200e3, sampling=0.08)
exit_wave  = plane_wave.multislice(potential).compute()
saed       = exit_wave.diffraction_patterns(max_angle=60).compute()

dp = np.asarray(saed.array)
fig, ax = plt.subplots(figsize=(6, 6))
# The bright centre is the DIRECT (undiffracted) beam -- ~100x brighter than the
# Bragg spots. show_diffraction clips it (like a beam stop) so the spots are visible.
show_diffraction(ax, dp, title="abTEM multislice — FCC [001] SAED (direct beam clipped)")
plt.show()
print("pattern shape:", dp.shape)
print("The four inner spots are the {200} family; the diagonal ones are {220}.")


## 3. Frozen phonons — the thermal-diffuse background

Real diffraction has a diffuse background from thermal atomic vibration (thermal-diffuse
scattering, TDS). Multislice reproduces it by averaging over several *frozen* snapshots of
displaced atoms. This background matters for analysis tools that background-subtract or fit
intensities — data without it is "too clean" and an analyzer may behave differently than on
real data. We use a few configs here for CPU; real work uses 16–32.

In [ ]:
# sigmas ~ 0.08-0.10 A is a typical RMS thermal displacement at room temperature for Au.
frozen = abtem.FrozenPhonons(atoms, num_configs=4, sigmas=0.1, seed=1)
pot_fp = abtem.Potential(frozen, sampling=0.10, slice_thickness=2.0, projection="infinite")

saed_fp = (abtem.PlaneWave(energy=200e3, sampling=0.10)
           .multislice(pot_fp)
           .diffraction_patterns(max_angle=50)
           .mean(0)                     # average over the phonon configurations
           .compute())

dp_fp = np.asarray(saed_fp.array)
fig, axs = plt.subplots(1, 2, figsize=(12, 6))
show_diffraction(axs[0], dp,    title="no TDS (single config)")
show_diffraction(axs[1], dp_fp, title=f"with TDS ({frozen.num_configs} frozen phonons)")
plt.tight_layout(); plt.show()
print("With frozen phonons, note the diffuse background between the spots (thermal-diffuse")
print("scattering) and the redistributed intensity -- absent in the single-config pattern.")


## 4. A small 4D-STEM scan → a portable 4D dataset

4D-STEM records a full diffraction pattern at every probe position, using a **convergent
probe** (a focused beam) rather than a plane wave. With a convergent probe each Bragg
reflection is no longer a sharp point — it spreads into a **disk** the size of the
convergence angle. So these patterns look different from the SAED spots above: they are
*CBED* (convergent-beam) patterns.

**A note on the convergence angle — this controls how the pattern looks.** If the
convergence semiangle is *smaller* than the spacing between reflections, the disks stay
separated and you see clean, spot-like disks. If it is *larger*, the disks **overlap** and
interference fringes appear in the overlaps — a "Ronchigram"-like pattern (this overlap is
exactly what ptychography reconstructs). For FCC Au at 200 kV the disks are ~10–12 mrad
apart, so we use **8 mrad** here to keep the disks separated and the pattern readable. Try
raising it to 20–30 mrad to see the overlapping-disk / ptychography regime.

We use a tiny **6×6** scan over a small region (kept small for CPU — real scans are
128×128+). The result is a 4D array `(scan_y, scan_x, det_y, det_x)` saved as a plain
`.npy`, which any analysis tool can open (py4DSTEM's `DataCube` wraps it directly).

In [ ]:
# Convergent probe. semiangle_cutoff sets the disk size:
#   8 mrad  < ~12 mrad disk spacing -> SEPARATED disks (clean, readable)
#   20-30 mrad                       -> OVERLAPPING disks (Ronchigram/ptychography)
CONV_MRAD = 8
probe = abtem.Probe(energy=200e3, semiangle_cutoff=CONV_MRAD, sampling=0.10)
pot_scan = abtem.Potential(atoms, sampling=0.10, slice_thickness=2.0, projection="infinite")
probe.grid.match(pot_scan)

# Scan a small region. We end just short of a full lattice repeat and scan a couple of
# unit cells so the sampled positions are representative (not all on one symmetry site).
scan_end = 2 * a_Au * (5/6)
scan = abtem.GridScan(start=(0, 0), end=(scan_end, scan_end), gpts=(6, 6))
detector = abtem.PixelatedDetector(max_angle=50)

dataset = probe.scan(pot_scan, scan=scan, detectors=detector).compute()
data_4d = np.asarray(dataset.array).astype(np.float32)   # (6, 6, det_y, det_x)
print(f"4D-STEM dataset shape: {data_4d.shape}  (convergence = {CONV_MRAD} mrad)")

np.save("fcc_4dstem.npy", data_4d)
print("saved -> fcc_4dstem.npy")

# A few CBED patterns across the scan (disks stay put; their intensities modulate).
fig, axs = plt.subplots(1, 4, figsize=(16, 4))
for ax, (iy, ix) in zip(axs, [(0,0),(0,3),(3,0),(5,5)]):
    show_diffraction(ax, data_4d[iy, ix], title=f"probe @ ({iy},{ix})", beamstop_radius=10)
plt.suptitle(f"CBED patterns at four probe positions (convergence {CONV_MRAD} mrad)")
plt.tight_layout(); plt.show()


## 5. Round-trip: analyse the simulated data as an external tool would

The point of the appendix: an analysis tool should treat the simulated 4D data like
experimental data. The two operations below — the **mean diffraction pattern** and a
**virtual bright-field** image — are exactly what a 4D-STEM analysis package does first,
and they need nothing beyond NumPy, so this runs with no extra dependency and no NumPy
conflict. (Section 5b shows the identical operations through py4DSTEM itself, isolated,
for anyone who wants the real library in the loop.)

In [ ]:
# Load the saved dataset (as any external tool would open a file).
data_4d = np.load("fcc_4dstem.npy")          # (scan_y, scan_x, det_y, det_x)
sy, sx, qy, qx = data_4d.shape
print("loaded 4D dataset:", data_4d.shape)

# (1) Mean diffraction pattern over all probe positions.
dp_mean = data_4d.mean(axis=(0, 1))

# (2) Virtual bright-field: integrate a disk around the direct beam at each position.
cy, cx = qy // 2, qx // 2
Y, X = np.mgrid[0:qy, 0:qx]
bf_disk = np.hypot(Y - cy, X - cx) <= 6
bf_image = (data_4d * bf_disk[None, None]).sum(axis=(2, 3))

fig, axs = plt.subplots(1, 2, figsize=(11, 5))
show_diffraction(axs[0], dp_mean, title="mean diffraction pattern", beamstop_radius=10)
axs[1].imshow(bf_image, cmap="gray")
axs[1].set_title(f"virtual bright-field ({sy}x{sx})"); axs[1].axis("off")
plt.tight_layout(); plt.show()
print("Round-trip complete: twin-style atoms -> abTEM multislice -> 4D dataset -> analysis.")


### 5b. (Optional) The same round-trip through py4DSTEM itself

py4DSTEM is the standard 4D-STEM analysis package, and it reads our saved array directly.
The **only** catch is packaging: py4DSTEM currently requires `numpy<2`, so installing it in
Colab downgrades NumPy and **requires a runtime restart** (and will show incompatibility
warnings for Colab's NumPy-2 packages — harmless for this isolated analysis). Run this
section **only if you want py4DSTEM specifically**, and treat it as a separate step:

1. Run the cell below to install py4DSTEM (it will downgrade NumPy).
2. **Runtime → Restart session** (required for the new NumPy to load).
3. Re-run the imports and the analysis cell (skip the abTEM sections — the `.npy` is already
   on disk).

If you just need the analysis, Section 5 above already did it with no downgrade.

In [ ]:
# --- OPTIONAL: only run if you specifically want py4DSTEM ---
# This downgrades NumPy to <2. Restart the runtime afterwards, then run the next cell.
# !pip -q install py4dstem
#
# After restarting the runtime, run:
#
# import numpy as np, py4DSTEM
# dc = py4DSTEM.DataCube(data=np.load("fcc_4dstem.npy"))
# dp_mean = dc.get_dp_mean()
# qy0, qx0 = dc.Qshape[0] // 2, dc.Qshape[1] // 2
# dc.get_virtual_image(mode="circle", geometry=((qy0, qx0), 6), name="BF")
# bf = dc.tree("BF").data
# # ... then plot dp_mean.data and bf as in Section 5.
print("Section 5b is optional and commented out by default (see the markdown above).")


## 6. Trying more samples

Everything above works for *any* structure — you just change the atoms. There are two
routes.

**Route A — different materials with ASE (self-contained).** `ase.build.bulk` knows the
common crystal structures, so one line gives you a new crystal. Different structures give
visibly different diffraction symmetry.

In [ ]:
from ase.build import bulk

more_samples = [
    ("Au FCC",     bulk("Au", "fcc",     a=4.05, cubic=True) * (6, 6, 20)),
    ("Fe BCC",     bulk("Fe", "bcc",     a=2.87, cubic=True) * (8, 8, 28)),
    ("Si diamond", bulk("Si", "diamond", a=5.43, cubic=True) * (5, 5, 15)),
    ("Cu FCC",     bulk("Cu", "fcc",     a=3.61, cubic=True) * (7, 7, 22)),
]

fig, axs = plt.subplots(1, len(more_samples), figsize=(4*len(more_samples), 4))
for ax, (name, aa) in zip(axs, more_samples):
    pot = abtem.Potential(aa, sampling=0.09, slice_thickness=2.0, projection="infinite")
    dp = (abtem.PlaneWave(energy=200e3, sampling=0.09)
          .multislice(pot).diffraction_patterns(max_angle=55).compute())
    show_diffraction(ax, np.asarray(dp.array), title=f"{name}\n{len(aa)} atoms")
plt.suptitle("Different crystals -> different SAED symmetry (multislice)")
plt.tight_layout(); plt.show()


**Route B — the *same* structures you built in the main twin.** This is the point of the
shared `get_atoms_in_region` interface: any twin sample (single crystals, the polycrystal,
the dislocation crystal, an Atomsk-loaded cell) hands back `(positions_Å, atomic_numbers)`,
which you wrap in an `ase.Atoms` and feed to the identical abTEM code. Then the dynamical
pattern corresponds to *your actual modeled object* — e.g. the multislice diffraction of the
specific dislocation or grain structure from the main notebook, not a generic crystal.

The snippet below is the bridge (commented out, since it needs the twin's `samples` package
on the path). Everything after the `atoms = ...` line is the same as Sections 2–5.

In [ ]:
# --- Bridge from a twin sample to abTEM (run inside the main twin environment) ---
# import sys; sys.path.append("path/to/your/twin")   # where the `samples` package lives
# import samples
# from ase import Atoms
#
# sample = samples.get_sample("dislocation_crystal")   # or bcc_single_crystal, polycrystal_grains, ...
# sample.generate_volume(40, 256, 256)                 # some samples need this first
# positions_A, Z = sample.get_atoms_in_region(cx_um=0, cy_um=0, half_width_um=0.02, depth_nm=12)
#
# # Wrap in an ASE Atoms with a periodic cell matching the returned extent.
# ext = positions_A.max(axis=0) - positions_A.min(axis=0)
# atoms_from_twin = Atoms(numbers=Z, positions=positions_A - positions_A.min(axis=0),
#                         cell=[ext[0], ext[1], ext[2]], pbc=True)
#
# # ...then run exactly the Section 2 / 3 / 4 code on `atoms_from_twin`.
print("Route B is a commented bridge: wrap twin get_atoms_in_region() output as ase.Atoms.")


## Notes, limits, and the GPU next step

**What you gained over the kinematical engine.** Physically correct spot *intensities*,
thickness-dependent dynamical behaviour, and (with frozen phonons) a realistic thermal-diffuse
background — the channels that quantitative analysis tools (thickness, atom-counting, strain,
ptychography) actually read. Positions were already correct kinematically; *intensities and
background* are what multislice adds.

**What is still on you (not the engine).** "As good as real" is capped by the **structure**,
not just the physics: an idealised procedural crystal gives the diffraction *of an idealised
crystal*. For studies where the object must be right (a specific dislocation core, a real strain
field, a textured polycrystal), feed in a realistic structure — an Atomsk build or an MD/relaxed
cell via ASE — using the same `atoms` entry point as Section 1. Also add **detector effects**
(camera point-spread/MTF, Poisson noise; the twin already models dose) on top of the simulated
pattern if your analyzer is sensitive to them.

**Validate the validator.** To *claim* the simulated data is analysis-grade, show the external
tool recovers a **known** ground truth from it (e.g. feed a known-thickness cell and confirm the
thickness is recovered). That test earns the claim.

**Why CPU is fine here but a GPU matters at scale.** Every step above is small on purpose. The
expensive case is a **full 4D scan** (128×128 positions) with **many phonon configs** — impractical
on CPU, but well within reach on a GPU. abTEM runs on GPU via CuPy (`abtem.config.set({"device":
"gpu"})`); Prismatic's PRISM algorithm is purpose-built to accelerate *scanned* STEM and can be
dramatically faster for large 4D datasets on CUDA hardware. The delay is a **generation-time** cost
paid once, after which the external analysis runs on the saved dataset at full speed. That is the
right place to spend the time — and it keeps the interactive twin fast for its actual job
(testing acquisition and decision logic).